# Fabric Notebook: EMP Map (wf_m_poc_xml_emp)
**Source:** XML → **Target:** CSV

**Created:** 2026-06-19

This notebook implements the PowerCenter EMP mapping for Fabric - load and transform employee data from XML to CSV.

## SECTION 1: IMPORTS & CONFIGURATION

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, 
    when, 
    trim, 
    upper, 
    cast,
    concat_ws,
    current_timestamp,
    count
)
from pyspark.sql.types import *
import os
from datetime import datetime

# Configuration
LAKEHOUSE_PATH = "/lakehouse/default/Files"
SOURCE_FILE = "employees.xml"
TARGET_LOCATION = "emp_poc_target"

print(f"📍 Map: EMP Source → Target")
print(f"⏰ Execution Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔹 Lakehouse Path: {LAKEHOUSE_PATH}")
print("=" * 80)

## SECTION 2: SOURCE EXTRACTION (XML Reader)

In [ ]:
print("\n📥 [STEP 1] Reading XML Source")
print("-" * 80)

try:
    # Read XML with schema inference
    df_source = spark.read \
        .format("xml") \
        .option("rowTag", "employee") \
        .option("inferSchema", "true") \
        .load(f"{LAKEHOUSE_PATH}/{SOURCE_FILE}")
    
    print(f"✓ XML Source loaded successfully")
    print(f"  Rows: {df_source.count()}")
    print(f"  Columns: {df_source.columns}")
    
    df_source.show(5)
    
except Exception as e:
    print(f"❌ Error reading XML source: {str(e)}")
    raise

## SECTION 3: DATA TRANSFORMATION (Map Logic)

In [ ]:
print("\n🔄 [STEP 2] Applying Transformations")
print("-" * 80)

try:
    # Define target schema (matches wf_m_poc_xml_emp output)
    target_schema = StructType([
        StructField("EMPLOYEE_ID", ByteType(), True),
        StructField("FIRST_NAME", StringType(), True),
        StructField("LAST_NAME", StringType(), True),
        StructField("SALARY", ShortType(), True),
        StructField("DEPARTMENT_ID", ByteType(), True),
    ])
    
    # Apply map transformations
    df_transformed = df_source.select(
        cast(col("EMPLOYEE_ID"), ByteType()).alias("EMPLOYEE_ID"),
        trim(upper(col("FIRST_NAME"))).alias("FIRST_NAME"),
        trim(upper(col("LAST_NAME"))).alias("LAST_NAME"),
        cast(col("SALARY"), ShortType()).alias("SALARY"),
        cast(col("DEPARTMENT_ID"), ByteType()).alias("DEPARTMENT_ID"),
    )
    
    print(f"✓ Transformations applied")
    print(f"  Rows processed: {df_transformed.count()}")
    print(f"  Data types verified")
    
    df_transformed.show(5)
    
    # Display schema
    print("\n📋 Target Schema:")
    df_transformed.printSchema()
    
except Exception as e:
    print(f"❌ Error applying transformations: {str(e)}")
    raise

## SECTION 4: DATA QUALITY CHECKS

In [ ]:
print("\n✅ [STEP 3] Data Quality Validation")
print("-" * 80)

try:
    # Check for nulls in key fields
    null_check = df_transformed.filter(
        (col("EMPLOYEE_ID").isNull()) | 
        (col("FIRST_NAME").isNull()) | 
        (col("LAST_NAME").isNull())
    ).count()
    
    print(f"  Null checks (key fields): {null_check} issues found")
    
    # Check for duplicates
    dup_check = df_transformed.count() - df_transformed.select("EMPLOYEE_ID").distinct().count()
    print(f"  Duplicate checks (EMPLOYEE_ID): {dup_check} duplicates found")
    
    # Salary range validation
    salary_check = df_transformed.filter(col("SALARY") < 0).count()
    print(f"  Salary validation (< 0): {salary_check} issues found")
    
    if null_check == 0 and dup_check == 0 and salary_check == 0:
        print(f"✓ All data quality checks PASSED")
    else:
        print(f"⚠️  Some data quality issues detected")
    
except Exception as e:
    print(f"❌ Error during validation: {str(e)}")
    raise

## SECTION 5: TARGET OUTPUT (CSV Writer)

In [ ]:
print("\n📤 [STEP 4] Writing to Target CSV")
print("-" * 80)

try:
    # Write to CSV
    target_path = f"{LAKEHOUSE_PATH}/{TARGET_LOCATION}"
    
    df_transformed.coalesce(1).write \
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .option("delimiter", ",") \
        .save(target_path)
    
    print(f"✓ Target CSV written successfully")
    print(f"  Location: {target_path}")
    print(f"  Records: {df_transformed.count()}")
    
except Exception as e:
    print(f"❌ Error writing target CSV: {str(e)}")
    raise

## SECTION 6: SUMMARY REPORT

In [ ]:
print("\n📊 [SUMMARY] Map Execution Report")
print("=" * 80)
print(f"Map Name:          wf_m_poc_xml_emp")
print(f"Source Format:     XML")
print(f"Target Format:     CSV")
print(f"Source Records:    {df_source.count()}")
print(f"Target Records:    {df_transformed.count()}")
print(f"Execution Status:  ✓ SUCCESS")
print(f"Timestamp:         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

## SECTION 7: CREATE DELTA TABLE (Optional)

In [ ]:
print("\n🔹 [STEP 6] Creating Delta Table (Optional)")
print("-" * 80)

try:
    # Drop existing table if exists
    spark.sql("DROP TABLE IF EXISTS emp_poc")
    
    # Create Delta table
    df_transformed.write.format("delta").mode("overwrite").saveAsTable("emp_poc")
    
    print(f"✓ Delta table 'emp_poc' created successfully")
    
    # Verify
    result_count = spark.sql("SELECT COUNT(*) as record_count FROM emp_poc").collect()[0][0]
    print(f"  Records in table: {result_count}")
    
except Exception as e:
    print(f"⚠️  Optional step skipped: {str(e)}")

print("\n✅ Map execution completed successfully!")